# Chapter 11: Docker Registries

---

If this topic is new to you, keep translating it back to three practical ideas: what job it does, when you would reach for it, and which nearby concepts it is easy to confuse with. That lens will make the details in this chapter feel connected instead of memorized.

## Why Docker Registries Matter in Real CI/CD Pipelines

In earlier chapters, you learned how to build Docker images and run containers locally. That is excellent for development, but teams need a **central place** to store, version, and share container images. That place is a **Docker registry**.

A registry is to container images what GitHub/GitLab is to source code:

- it stores artifacts,
- tracks versions (tags and digests),
- enables collaboration,
- and acts as the source of truth for deployments.

Without a registry, every deployment would require building images from source directly on production infrastructure, which is slow, fragile, and hard to audit.

---

## Core Concepts You Must Understand

### 1) Repository, Tag, and Digest

- **Repository**: A named collection of images, for example: `myorg/payment-service`.
- **Tag**: A mutable label, for example: `v1.8.2`, `staging`, or `latest`.
- **Digest**: An immutable content hash, for example: `sha256:...`.

> Best practice: use tags for readability, but prefer digests for production pinning when strict reproducibility matters.
### 2) Public vs Private Registries

Sections like this become clearer when you separate capabilities from tradeoffs. 

- **Public registry** (like Docker Hub public repos): useful for open images.
- **Private registry** (Docker Hub private repos, GHCR, ECR, GCR, ACR, self-hosted Harbor): required for proprietary code and internal services.
### 3) Authentication and Authorization

Registries enforce access through credentials or tokens. In CI/CD this usually means:

- short-lived tokens,
- robot/service accounts,
- and least-privilege permissions (`pull` for runtime, `push` only in build pipelines).

---

## Common Registry Providers

- **Docker Hub**: default ecosystem registry, easy to start.
- **GitHub Container Registry (GHCR)**: integrates with GitHub Actions and repo permissions.
- **AWS ECR**: deep AWS IAM integration.
- **Google Artifact Registry / GCR**: Google Cloud-native integration.
- **Azure Container Registry (ACR)**: first-class Azure integration.
- **Harbor**: self-hosted enterprise registry with scanning and policy controls.

Choose a registry that matches your platform, governance requirements, and team workflows.

---
## Typical CI/CD Workflow with a Registry

1. Developer pushes code.
2. CI builds Docker image.
3. CI runs tests/security scans.
4. CI tags image (for example, `1.4.0` and commit SHA).
5. CI pushes image to registry.
6. CD deploys by pulling the exact image from registry.

This separation is powerful: **build once, deploy many times** across dev/staging/prod.

---
## Tagging Strategy That Scales

A practical strategy is to publish multiple tags per build:

- `myorg/app:1.9.0` (semantic release)
- `myorg/app:main-2026-03-16` (branch/date context)
- `myorg/app:git-<shortsha>` (traceability)

Avoid relying only on `latest` in automation. It is convenient but ambiguous and can cause accidental rollbacks/roll-forwards.

---

## Security and Governance Essentials

Registries are part of your software supply chain. Treat them as security-critical.

- Enable vulnerability scanning for pushed images.
- Sign images (e.g., Sigstore/Cosign) and verify before deployment.
- Use immutable tags for release channels where possible.
- Implement retention policies to clean old artifacts while preserving important releases.
- Enforce access controls and audit logs.

---

## Example Commands

In [ ]:
# Authenticate
Docker login ghcr.io

# Tag image for registry
docker tag payment-service:local ghcr.io/myorg/payment-service:v1.0.0

# Push
docker push ghcr.io/myorg/payment-service:v1.0.0

# Pull on another machine/environment
docker pull ghcr.io/myorg/payment-service:v1.0.0


---

## Real-World Pitfalls (and How to Avoid Them)

- **Pitfall:** Overwriting tags accidentally.  
  **Fix:** use immutable release tags and controlled promotion.

- **Pitfall:** CI pipelines with long-lived static credentials.  
  **Fix:** use short-lived OIDC-based auth when supported.

- **Pitfall:** Images cannot be traced back to source commit.  
  **Fix:** include commit SHA tags and OCI labels (`org.opencontainers.image.revision`).

- **Pitfall:** Registry costs grow due to unlimited artifact retention.  
  **Fix:** apply lifecycle/retention policies per repository.

---
## Summary

Docker registries are the backbone of containerized CI/CD delivery. They provide a secure, auditable, and repeatable mechanism to distribute application artifacts across environments. When combined with strong tagging, scanning, and access control practices, registries significantly improve both delivery speed and operational safety.


<div style='width:100%; display:flex; justify-content:space-between; align-items:center; margin: 1em 0;'>
  <a href='10. docker_security_best_practices.ipynb' style='font-weight:bold; font-size:1.05em;'>&larr; Previous</a>
  <a href='../TOC.md' style='font-weight:bold; font-size:1.05em; text-align:center;'>Table of Contents</a>
  <a href='../3. kubernetes_fundamentals/12. kubernetes_architecture.ipynb' style='font-weight:bold; font-size:1.05em;'>Next &rarr;</a>
</div>
